# Final paper analyzer


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 320)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root.")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"

SS_DIR = RESULTS_ROOT / "experiment_2026_09_30_final_unregularized_single_shot"
AB_DIR = RESULTS_ROOT / "experiment_2026_09_30_final_unregularized_abundant"
BASELINE_DIR = RESULTS_ROOT / "experiment_2026_09_28_true_graph_static_and_analytic_baselines"

OUT_DIR = RESULTS_ROOT / "paper_analysis_2026_09_30_final_unregularized"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

required = [
    SS_DIR / "COMPUTATION_COMPLETE.json",
    SS_DIR / "RUN_MANIFEST.json",
    SS_DIR / "combined_summary.csv",
    AB_DIR / "COMPUTATION_COMPLETE.json",
    AB_DIR / "combined_summary.csv",
    BASELINE_DIR / "COMPUTATION_COMPLETE.json",
    BASELINE_DIR / "true_graph_static_all_195.csv",
    BASELINE_DIR / "true_graph_analytic_supported_90.csv",
    BASELINE_DIR / "analytic_centrality_uniqueness_diagnostics.csv",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)

ss_complete = json.loads((SS_DIR / "COMPUTATION_COMPLETE.json").read_text(encoding="utf-8"))
ab_complete = json.loads((AB_DIR / "COMPUTATION_COMPLETE.json").read_text(encoding="utf-8"))
base_complete = json.loads((BASELINE_DIR / "COMPUTATION_COMPLETE.json").read_text(encoding="utf-8"))
ss_manifest = json.loads((SS_DIR / "RUN_MANIFEST.json").read_text(encoding="utf-8"))

assert ss_complete["status"] == "success"
assert ab_complete["status"] == "success"
assert base_complete["status"] == "success"
assert ss_complete["study_name"] == "final_unregularized_single_shot_2026_09_30"
assert ab_complete["study_name"] == "final_unregularized_abundant_2026_09_30"
assert base_complete["study_name"] == "true_graph_static_and_analytic_baselines_2026_09_28"
assert ss_complete["n_environments"] == 195
assert ss_complete["n_learned_runs"] == 585
assert ab_complete["n_environments"] == 195
assert ab_complete["n_learned_runs"] == 585
assert ss_manifest["scientific_config"]["regularization"] == "none"

print("Validated final unregularized single-shot, abundant, and true-model baseline results.")


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_118988\60872266.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Validated final unregularized single-shot, abundant, and true-model baseline results.


In [2]:
ss_raw = pd.read_csv(SS_DIR / "combined_summary.csv")
ab_raw = pd.read_csv(AB_DIR / "combined_summary.csv")
static_raw = pd.read_csv(BASELINE_DIR / "true_graph_static_all_195.csv")
analytic_raw = pd.read_csv(BASELINE_DIR / "true_graph_analytic_supported_90.csv")
analytic_diag = pd.read_csv(BASELINE_DIR / "analytic_centrality_uniqueness_diagnostics.csv")

assert len(ss_raw) == 1170
assert len(ab_raw) == 1170

SS_LEARNED = "learned_nonlinear"
AB_LEARNED = "learned_abundant_nonlinear"

assert len(ss_raw[ss_raw["policy"] == SS_LEARNED]) == 585
assert len(ab_raw[ab_raw["policy"] == AB_LEARNED]) == 585
assert ss_raw[ss_raw["policy"] == SS_LEARNED]["environment_id"].nunique() == 195
assert ab_raw[ab_raw["policy"] == AB_LEARNED]["environment_id"].nunique() == 195

RECOVERY_METRICS = [
    "A_MAE",
    "A_Fro",
    "row_TV",
    "true_edge_mass",
    "support_topdegree_recall",
    "edge_AP",
    "centrality_L1",
    "row_entropy_norm",
    "effective_neighbors",
]

def learned_env_table(df, policy, regime):
    x = df[df["policy"] == policy].copy()
    control_col = "final_mean" if regime == "single_shot" else "mean_end"
    prediction_col = (
        "final_fit_model_over_identity"
        if regime == "single_shot"
        else "model_over_identity"
    )

    needed = [
        "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed",
        control_col, prediction_col, *RECOVERY_METRICS,
    ]
    missing = [c for c in needed if c not in x.columns]
    if missing:
        raise RuntimeError(f"{regime} final result is missing columns: {missing}")

    agg = {
        control_col: "mean",
        prediction_col: "mean",
        **{m: "mean" for m in RECOVERY_METRICS},
    }
    meta = [
        "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed",
    ]
    for m in meta:
        agg[m] = "first"

    out = x.groupby("environment_id", as_index=False).agg(agg)
    out = out.rename(columns={
        control_col: "control",
        prediction_col: "prediction_ratio",
    })
    out["regime"] = regime
    return out


ss_env = learned_env_table(ss_raw, SS_LEARNED, "single_shot")
ab_env = learned_env_table(ab_raw, AB_LEARNED, "abundant")

assert len(ss_env) == 195
assert len(ab_env) == 195

def baseline_table(df, metric):
    x = df[df["policy"].isin(["no_control", "uniform", "true_graph"])].copy()
    x = x[[
        "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed", "policy", metric,
    ]]
    return x.pivot(index="environment_id", columns="policy", values=metric).reset_index()

ss_baseline = baseline_table(ss_raw, "final_mean")
ab_baseline = baseline_table(ab_raw, "mean_end")

baseline_check = ss_baseline.merge(
    ab_baseline,
    on="environment_id",
    suffixes=("_ss", "_ab"),
    validate="one_to_one",
)
for p in ["no_control", "uniform", "true_graph"]:
    assert np.allclose(
        baseline_check[f"{p}_ss"],
        baseline_check[f"{p}_ab"],
        atol=1e-12,
        rtol=0.0,
    )

static_check = (
    ss_raw[ss_raw["policy"] == "true_graph"][
        ["environment_id", "final_mean"]
    ]
    .drop_duplicates()
    .merge(
        static_raw[["environment_id", "final_mean"]],
        on="environment_id",
        suffixes=("_final", "_baseline"),
        validate="one_to_one",
    )
)
assert np.allclose(
    static_check["final_mean_final"],
    static_check["final_mean_baseline"],
    atol=1e-12,
    rtol=0.0,
)

print("Final learned rows and deterministic baselines validated.")


Final learned rows and deterministic baselines validated.


In [3]:
def mean_ci95(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    mean = float(values.mean()) if n else np.nan
    if n < 2:
        return mean, np.nan
    half = 1.96 * float(values.std(ddof=1)) / math.sqrt(n)
    return mean, half


base_env = (
    ss_raw[ss_raw["policy"].isin(["no_control", "uniform", "true_graph"])]
    [["environment_id", "scenario_class", "family", "dynamics", "policy", "final_mean"]]
    .drop_duplicates()
    .pivot(
        index=["environment_id", "scenario_class", "family", "dynamics"],
        columns="policy",
        values="final_mean",
    )
    .reset_index()
)

control_env = (
    base_env
    .merge(
        ss_env[["environment_id", "control"]].rename(
            columns={"control": "learned_single_shot"}
        ),
        on="environment_id",
        validate="one_to_one",
    )
    .merge(
        ab_env[["environment_id", "control"]].rename(
            columns={"control": "learned_abundant"}
        ),
        on="environment_id",
        validate="one_to_one",
    )
)

analytic = analytic_raw[[
    "environment_id", "final_mean",
    "max_laplacian_nullity",
    "fraction_campaigns_nonunique_centrality",
]].rename(columns={"final_mean": "true_graph_analytic"})

control_env = control_env.merge(
    analytic,
    on="environment_id",
    how="left",
    validate="one_to_one",
)

for col in ["learned_single_shot", "learned_abundant"]:
    control_env[f"{col}_minus_uniform"] = control_env[col] - control_env["uniform"]
    control_env[f"{col}_minus_true_graph_static"] = control_env[col] - control_env["true_graph"]
    control_env[f"{col}_minus_true_graph_analytic"] = control_env[col] - control_env["true_graph_analytic"]

control_env.to_csv(TABLE_DIR / "paper_control_environment_level.csv", index=False)

group_rows = []
for keys, g in control_env.groupby(["scenario_class", "dynamics"], sort=False):
    row = {
        "scenario_class": keys[0],
        "dynamics": keys[1],
        "n_environments": len(g),
        "uniform": g["uniform"].mean(),
        "true_graph_static": g["true_graph"].mean(),
        "learned_single_shot": g["learned_single_shot"].mean(),
        "learned_abundant": g["learned_abundant"].mean(),
        "true_graph_analytic": g["true_graph_analytic"].mean(),
        "analytic_n": int(g["true_graph_analytic"].notna().sum()),
        "analytic_nonunique_fraction": g["fraction_campaigns_nonunique_centrality"].mean(),
        "analytic_max_nullity": g["max_laplacian_nullity"].max(),
    }
    for learned in ["learned_single_shot", "learned_abundant"]:
        row[f"{learned}_minus_uniform"] = (g[learned] - g["uniform"]).mean()
        row[f"{learned}_minus_static"] = (g[learned] - g["true_graph"]).mean()
        valid = g["true_graph_analytic"].notna()
        row[f"{learned}_minus_analytic"] = (
            (g.loc[valid, learned] - g.loc[valid, "true_graph_analytic"]).mean()
            if valid.any() else np.nan
        )
        _, ci = mean_ci95(g[learned])
        row[f"{learned}_ci95"] = ci
    group_rows.append(row)

control_summary = pd.DataFrame(group_rows)
control_summary.to_csv(TABLE_DIR / "paper_control_summary_by_dynamics.csv", index=False)


family_rows = []
for keys, g in control_env.groupby(
    ["scenario_class", "dynamics", "family"],
    sort=False,
):
    family_rows.append({
        "scenario_class": keys[0],
        "dynamics": keys[1],
        "family": keys[2],
        "n_environments": len(g),
        "uniform": g["uniform"].mean(),
        "true_graph_static": g["true_graph"].mean(),
        "learned_single_shot": g["learned_single_shot"].mean(),
        "learned_abundant": g["learned_abundant"].mean(),
        "true_graph_analytic": g["true_graph_analytic"].mean(),
        "single_shot_minus_uniform": (
            g["learned_single_shot"] - g["uniform"]
        ).mean(),
        "abundant_minus_uniform": (
            g["learned_abundant"] - g["uniform"]
        ).mean(),
        "single_shot_minus_static": (
            g["learned_single_shot"] - g["true_graph"]
        ).mean(),
        "abundant_minus_static": (
            g["learned_abundant"] - g["true_graph"]
        ).mean(),
    })

family_summary = pd.DataFrame(family_rows)
family_summary.to_csv(
    TABLE_DIR / "paper_control_summary_by_family.csv",
    index=False,
)

overall = []
for regime, col in [
    ("single_shot", "learned_single_shot"),
    ("abundant", "learned_abundant"),
]:
    g = control_env
    mean, ci = mean_ci95(g[col])
    overall.append({
        "regime": regime,
        "n_environments": len(g),
        "learned_control": mean,
        "learned_control_ci95": ci,
        "uniform": g["uniform"].mean(),
        "true_graph_static": g["true_graph"].mean(),
        "learned_minus_uniform": (g[col] - g["uniform"]).mean(),
        "learned_minus_true_graph_static": (g[col] - g["true_graph"]).mean(),
    })

overall_control = pd.DataFrame(overall)
overall_control.to_csv(TABLE_DIR / "paper_control_overall.csv", index=False)

display(overall_control.round(6))
display(control_summary.round(6))


,regime,n_environments,learned_control,learned_control_ci95,uniform,true_graph_static,learned_minus_uniform,learned_minus_true_graph_static
0,single_shot,195,0.763732,0.012071,0.640688,0.784534,0.123044,-0.020803
1,abundant,195,0.767386,0.012325,0.640688,0.784534,0.126698,-0.017148


,scenario_class,dynamics,n_environments,uniform,true_graph_static,learned_single_shot,learned_abundant,true_graph_analytic,analytic_n,analytic_nonunique_fraction,analytic_max_nullity,learned_single_shot_minus_uniform,learned_single_shot_minus_static,learned_single_shot_minus_analytic,learned_single_shot_ci95,learned_abundant_minus_uniform,learned_abundant_minus_static,learned_abundant_minus_analytic,learned_abundant_ci95
0,structured_mechanism,friedkinjohnsen,15,0.417208,0.613842,0.611277,0.613674,NaN,0,NaN,NaN,0.194070,-0.002564,NaN,0.003663,0.196466,-0.000168,NaN,0.003459
1,structured_mechanism,hegselmannkrause,15,0.669797,0.895423,0.876433,0.753066,NaN,0,NaN,NaN,0.206636,-0.018990,NaN,0.002527,0.083269,-0.142358,NaN,0.010866
2,structured_mechanism,nonlinearinfluence,15,0.527619,0.865575,0.864993,0.865087,0.865575,15,0.000,1.0,0.337374,-0.000582,-0.000582,0.004449,0.337468,-0.000489,-0.000489,0.004848
3,unstructured_random,coca,25,0.736536,0.781852,0.763968,0.777333,0.643016,25,0.676,2.0,0.027432,-0.017884,0.120952,0.014900,0.040797,-0.004519,0.134317,0.015766
4,unstructured_random,friedkinjohnsen,25,0.608784,0.695590,0.684417,0.685874,NaN,0,NaN,NaN,0.075632,-0.011173,NaN,0.023074,0.077089,-0.009716,NaN,0.023474
5,unstructured_random,hegselmannkrause,25,0.685553,0.821060,0.790234,0.833845,NaN,0,NaN,NaN,0.104682,-0.030825,NaN,0.025799,0.148292,0.012785,NaN,0.023436
6,unstructured_random,laplacian,25,0.693651,0.816446,0.763632,0.813420,0.816446,25,0.000,1.0,0.069980,-0.052815,-0.052815,0.020875,0.119769,-0.003026,-0.003026,0.020403
7,unstructured_random,nonlinearinfluence,25,0.693128,0.844726,0.820113,0.834424,0.844771,25,0.000,1.0,0.126986,-0.024613,-0.024657,0.022852,0.141296,-0.010302,-0.010346,0.023575
8,unstructured_random,repulsion,25,0.610938,0.734787,0.723120,0.701620,NaN,0,NaN,NaN,0.112181,-0.011667,NaN,0.021600,0.090682,-0.033167,NaN,0.021437


In [4]:
recovery = pd.concat([ss_env, ab_env], ignore_index=True)

recovery_rows = []
for keys, g in recovery.groupby(["regime", "scenario_class", "dynamics"], sort=False):
    row = {
        "regime": keys[0],
        "scenario_class": keys[1],
        "dynamics": keys[2],
        "n_environments": len(g),
        "control": g["control"].mean(),
        "prediction_ratio": g["prediction_ratio"].mean(),
    }
    for m in RECOVERY_METRICS:
        row[m] = g[m].mean()
    recovery_rows.append(row)

recovery_summary = pd.DataFrame(recovery_rows)
recovery_summary.to_csv(TABLE_DIR / "paper_A_recovery_by_dynamics.csv", index=False)

recovery_overall = (
    recovery.groupby("regime", as_index=False)
    .agg(
        n_environments=("environment_id", "size"),
        control=("control", "mean"),
        prediction_ratio=("prediction_ratio", "mean"),
        A_MAE=("A_MAE", "mean"),
        A_Fro=("A_Fro", "mean"),
        row_TV=("row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        support_topdegree_recall=("support_topdegree_recall", "mean"),
        edge_AP=("edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        row_entropy_norm=("row_entropy_norm", "mean"),
        effective_neighbors=("effective_neighbors", "mean"),
    )
)
recovery_overall.to_csv(TABLE_DIR / "paper_A_recovery_overall.csv", index=False)

seed_rows = []
for regime, raw, policy, control_col in [
    ("single_shot", ss_raw, SS_LEARNED, "final_mean"),
    ("abundant", ab_raw, AB_LEARNED, "mean_end"),
]:
    x = raw[raw["policy"] == policy]
    for seed, g in x.groupby("learning_seed"):
        seed_rows.append({
            "regime": regime,
            "learning_seed": int(seed),
            "control": g[control_col].mean(),
            "row_TV": g["row_TV"].mean(),
            "true_edge_mass": g["true_edge_mass"].mean(),
            "edge_AP": g["edge_AP"].mean(),
            "centrality_L1": g["centrality_L1"].mean(),
        })

seed_summary = pd.DataFrame(seed_rows)
seed_summary.to_csv(TABLE_DIR / "paper_seed_robustness.csv", index=False)

display(recovery_overall.round(6))


,regime,n_environments,control,prediction_ratio,A_MAE,A_Fro,row_TV,true_edge_mass,support_topdegree_recall,edge_AP,centrality_L1,row_entropy_norm,effective_neighbors
0,abundant,195,0.767386,0.632985,0.076514,1.973094,0.573853,0.435848,0.735423,0.745979,0.494662,0.902105,10.985173
1,single_shot,195,0.763732,0.173999,0.050788,1.547163,0.380909,0.660697,0.697701,0.708441,0.497814,0.547512,5.107016


In [5]:
analytic_env = (
    analytic_raw[[
        "environment_id", "scenario_class", "family", "dynamics",
        "final_mean", "mean_over_campaigns",
        "max_laplacian_nullity",
        "fraction_campaigns_nonunique_centrality",
    ]]
    .rename(columns={
        "final_mean": "analytic_final",
        "mean_over_campaigns": "analytic_path",
    })
    .merge(
        static_raw[["environment_id", "final_mean", "mean_over_campaigns"]]
        .rename(columns={
            "final_mean": "static_final",
            "mean_over_campaigns": "static_path",
        }),
        on="environment_id",
        validate="one_to_one",
    )
    .merge(
        ss_env[["environment_id", "control"]]
        .rename(columns={"control": "single_shot_final"}),
        on="environment_id",
        validate="one_to_one",
    )
    .merge(
        ab_env[["environment_id", "control"]]
        .rename(columns={"control": "abundant_final"}),
        on="environment_id",
        validate="one_to_one",
    )
)

analytic_rows = []
for keys, g in analytic_env.groupby(["scenario_class", "dynamics"], sort=False):
    analytic_rows.append({
        "scenario_class": keys[0],
        "dynamics": keys[1],
        "n_environments": len(g),
        "static_final": g["static_final"].mean(),
        "analytic_final": g["analytic_final"].mean(),
        "single_shot_final": g["single_shot_final"].mean(),
        "abundant_final": g["abundant_final"].mean(),
        "single_shot_minus_analytic": (g["single_shot_final"] - g["analytic_final"]).mean(),
        "abundant_minus_analytic": (g["abundant_final"] - g["analytic_final"]).mean(),
        "analytic_minus_static": (g["analytic_final"] - g["static_final"]).mean(),
        "max_nullity": int(g["max_laplacian_nullity"].max()),
        "mean_fraction_nonunique": g["fraction_campaigns_nonunique_centrality"].mean(),
    })

analytic_summary = pd.DataFrame(analytic_rows)
analytic_env.to_csv(TABLE_DIR / "paper_analytic_baseline_environment_level.csv", index=False)
analytic_summary.to_csv(TABLE_DIR / "paper_analytic_baseline_summary.csv", index=False)

print("Analytic true-model baseline:")
display(analytic_summary.round(6))

if (analytic_summary["mean_fraction_nonunique"].fillna(0) > 0).any():
    print("NOTE: at least one analytic baseline has non-unique Laplacian centrality; see diagnostics before using it as a paper reference.")


Analytic true-model baseline:


,scenario_class,dynamics,n_environments,static_final,analytic_final,single_shot_final,abundant_final,single_shot_minus_analytic,abundant_minus_analytic,analytic_minus_static,max_nullity,mean_fraction_nonunique
0,unstructured_random,laplacian,25,0.816446,0.816446,0.763632,0.813420,-0.052815,-0.003026,0.000000,1,0.000
1,unstructured_random,coca,25,0.781852,0.643016,0.763968,0.777333,0.120952,0.134317,-0.138836,2,0.676
2,unstructured_random,nonlinearinfluence,25,0.844726,0.844771,0.820113,0.834424,-0.024657,-0.010346,0.000044,1,0.000
3,structured_mechanism,nonlinearinfluence,15,0.865575,0.865575,0.864993,0.865087,-0.000582,-0.000489,0.000000,1,0.000


NOTE: at least one analytic baseline has non-unique Laplacian centrality; see diagnostics before using it as a paper reference.


In [6]:
DYNAMICS_LABELS = {
    "laplacian": "Linear",
    "coca": "COCA",
    "hegselmannkrause": "HK",
    "friedkinjohnsen": "FJ",
    "nonlinearinfluence": "NLI",
    "repulsion": "Repulsion",
}

def save_control_figure(scenario_class, filename):
    xdf = control_summary[
        control_summary["scenario_class"] == scenario_class
    ].copy()

    order = [
        "laplacian", "coca", "hegselmannkrause",
        "friedkinjohnsen", "nonlinearinfluence", "repulsion",
    ]
    if scenario_class == "structured_mechanism":
        order = ["hegselmannkrause", "friedkinjohnsen", "nonlinearinfluence"]

    xdf["order"] = xdf["dynamics"].map({d:i for i,d in enumerate(order)})
    xdf = xdf.sort_values("order")
    labels = [DYNAMICS_LABELS.get(d, d) for d in xdf["dynamics"]]

    series = [
        ("Uniform", "uniform"),
        ("Single-shot", "learned_single_shot"),
        ("Abundant", "learned_abundant"),
        ("True A (static)", "true_graph_static"),
        ("True model (analytic)", "true_graph_analytic"),
    ]

    pos = np.arange(len(xdf))
    width = 0.16
    fig, ax = plt.subplots(figsize=(max(7.0, 1.25 * len(xdf)), 4.6))

    for k, (label, col) in enumerate(series):
        offset = (k - (len(series)-1)/2) * width
        vals = xdf[col].to_numpy(dtype=float)
        ax.bar(pos + offset, vals, width=width, label=label)

    ax.set_xticks(pos)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Final mean opinion")
    ax.set_ylim(0, 1)
    ax.legend(ncol=2, fontsize=8)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{filename}.png", dpi=220, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.close(fig)


save_control_figure("unstructured_random", "fig_control_unstructured")
save_control_figure("structured_mechanism", "fig_control_structured")

plot_rec = recovery_summary.copy()
plot_rec["label"] = (
    plot_rec["scenario_class"].map({
        "unstructured_random": "Generic",
        "structured_mechanism": "Structured",
    })
    + " "
    + plot_rec["dynamics"].map(DYNAMICS_LABELS).fillna(plot_rec["dynamics"])
)

label_order = (
    plot_rec[["scenario_class", "dynamics", "label"]]
    .drop_duplicates()
    .sort_values(["scenario_class", "dynamics"])["label"]
    .tolist()
)

pivot = plot_rec.pivot(index="label", columns="regime", values="row_TV").reindex(label_order)
fig, ax = plt.subplots(figsize=(10.5, 4.8))
pos = np.arange(len(pivot))
width = 0.36
ax.bar(pos - width/2, pivot["single_shot"], width=width, label="Single-shot")
ax.bar(pos + width/2, pivot["abundant"], width=width, label="Abundant")
ax.set_xticks(pos)
ax.set_xticklabels(pivot.index, rotation=35, ha="right")
ax.set_ylabel("Weighted row-TV error")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_A_recovery_rowTV.png", dpi=220, bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_A_recovery_rowTV.pdf", bbox_inches="tight")
plt.close(fig)

print("Figures:", FIG_DIR)


Figures: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\paper_analysis_2026_09_30_final_unregularized\figures


In [7]:
def latex_table(df, path, columns=None, float_format="%.4f"):
    x = df if columns is None else df[columns]
    path.write_text(
        x.to_latex(index=False, float_format=float_format, na_rep="--"),
        encoding="utf-8",
    )

latex_table(
    overall_control,
    TABLE_DIR / "paper_control_overall.tex",
)
latex_table(
    recovery_overall,
    TABLE_DIR / "paper_A_recovery_overall.tex",
)

headline = []
for _, r in overall_control.iterrows():
    headline.append(
        f"{r['regime']}: learned={r['learned_control']:.6f}, "
        f"uniform={r['uniform']:.6f}, "
        f"true_A_static={r['true_graph_static']:.6f}, "
        f"learned-uniform={r['learned_minus_uniform']:+.6f}, "
        f"learned-true_A_static={r['learned_minus_true_graph_static']:+.6f}"
    )

for _, r in recovery_overall.iterrows():
    headline.append(
        f"{r['regime']} A recovery: "
        f"row_TV={r['row_TV']:.6f}, "
        f"A_MAE={r['A_MAE']:.6f}, "
        f"true_edge_mass={r['true_edge_mass']:.6f}, "
        f"edge_AP={r['edge_AP']:.6f}, "
        f"centrality_L1={r['centrality_L1']:.6f}"
    )

coca = analytic_summary[analytic_summary["dynamics"] == "coca"]
if len(coca):
    r = coca.iloc[0]
    headline.append(
        f"analytic COCA diagnostic: mean_fraction_nonunique="
        f"{r['mean_fraction_nonunique']:.6f}, max_nullity={int(r['max_nullity'])}"
    )

(OUT_DIR / "paper_headline_numbers.txt").write_text(
    "\n".join(headline) + "\n",
    encoding="utf-8",
)

print("\n".join(headline))
print("\nOutputs:", OUT_DIR)
print("FINAL PAPER ANALYSIS COMPLETE")


single_shot: learned=0.763732, uniform=0.640688, true_A_static=0.784534, learned-uniform=+0.123044, learned-true_A_static=-0.020803
abundant: learned=0.767386, uniform=0.640688, true_A_static=0.784534, learned-uniform=+0.126698, learned-true_A_static=-0.017148
abundant A recovery: row_TV=0.573853, A_MAE=0.076514, true_edge_mass=0.435848, edge_AP=0.745979, centrality_L1=0.494662
single_shot A recovery: row_TV=0.380909, A_MAE=0.050788, true_edge_mass=0.660697, edge_AP=0.708441, centrality_L1=0.497814
analytic COCA diagnostic: mean_fraction_nonunique=0.676000, max_nullity=2

Outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\paper_analysis_2026_09_30_final_unregularized
FINAL PAPER ANALYSIS COMPLETE
